In [31]:
import pandas as pd
from pathlib import Path

RAW_DIR    = Path("../raw_data")
PROCESSED  = Path("../processed")
REF_DIR    = Path("../reference")
PROCESSED.mkdir(exist_ok=True)

# These are the 28 column names matching the raw file structure exactly
# Sr.No + Bank Name + 26 data columns — same order as the Excel sheet
RAW_COLS = [
    "sr_no", "bank_name",
    "atm_onsite", "atm_offsite", "pos_terminals", "micro_atms",
    "bharat_qr", "upi_qr",
    "cc_outstanding", "dc_outstanding",
    "cc_pos_vol", "cc_pos_val",
    "cc_online_vol", "cc_online_val",
    "cc_others_vol", "cc_others_val",
    "cc_atm_cash_vol", "cc_atm_cash_val",
    "dc_pos_vol", "dc_pos_val",
    "dc_online_vol", "dc_online_val",
    "dc_others_vol", "dc_others_val",
    "dc_atm_cash_vol", "dc_atm_cash_val",
    "dc_pos_cash_vol", "dc_pos_cash_val"
]

In [37]:
bank_master = pd.read_excel("../reference/bank_name_master.xlsx")

# Build two lookup dictionaries from the master table
# Key = raw name exactly as it appears in files
# Value = standardized name / category
name_to_std = dict(zip(bank_master["bank_name_raw"], bank_master["bank_name_std"]))
name_to_cat = dict(zip(bank_master["bank_name_raw"], bank_master["bank_category"]))

print(f"Master loaded: {len(bank_master)} name mappings")

Master loaded: 91 name mappings


In [43]:
all_months = []   # collects cleaned DataFrame from each file
unmatched  = []   # collects any bank names not found in master

for filepath in sorted(RAW_DIR.glob("rbi_*.xlsx")):

    # Step 1: Extract year and month from filename
    # rbi_2022_03.xlsx → parts = ['rbi', '2022', '03']
    parts = filepath.stem.split("_")
    year  = int(parts[1])
    month = int(parts[2])

    # Step 2: Read raw Excel
    # skiprows=7 skips: title row + 5 header rows + column numbers row
    # header=None tells pandas not to treat any row as column names
    # names=RAW_COLS assigns our clean names directly
    df = pd.read_excel(filepath, header=None, skiprows=7, names=RAW_COLS)

    # Step 3: Keep only valid bank rows
    # Category sub-headers (Public Sector Banks etc.) and Total row
    # have no numeric sr_no — this single line removes all of them
    df["sr_no"] = pd.to_numeric(df["sr_no"], errors="coerce")
    df = df[df["sr_no"].notna() & (df["sr_no"] > 0)].copy()

    # Step 4: Strip whitespace from bank names
    # Catches leading/trailing spaces like " SBM BANK INDIA LTD"
    df["bank_name"] = df["bank_name"].astype(str).str.strip()

    # Step 5: Standardize bank names using master lookup
    df["bank_name_std"] = df["bank_name"].map(name_to_std)
    df["bank_category"] = df["bank_name"].map(name_to_cat)

    # Step 6: Log any names not found in master
    # These need to be added to bank_name_master.xlsx manually
    missing = df[df["bank_name_std"].isna()][["bank_name"]].copy()
    if not missing.empty:
        missing["file"] = filepath.name
        unmatched.append(missing)

    # Step 7: Add date columns
    df["month"] = month
    df["year"]  = year

    # Step 8: Convert all data columns to numeric
    # errors="coerce" turns any non-numeric cell into NaN
    # fillna(0) replaces NaN with 0 (genuine zero — bank had no transactions)
    # Infrastructure columns — keep NaN for missing source data
    infra_cols = [
        "atm_onsite", "atm_offsite", "pos_terminals",
        "micro_atms", "bharat_qr", "upi_qr",
        "cc_outstanding", "dc_outstanding"
    ]

    # Transaction columns — fill NaN with 0 (no activity = zero)
    transaction_cols = [
        "cc_pos_vol", "cc_pos_val",
        "cc_online_vol", "cc_online_val",
        "cc_others_vol", "cc_others_val",
        "cc_atm_cash_vol", "cc_atm_cash_val",
        "dc_pos_vol", "dc_pos_val",
        "dc_online_vol", "dc_online_val",
        "dc_others_vol", "dc_others_val",
        "dc_atm_cash_vol", "dc_atm_cash_val",
        "dc_pos_cash_vol", "dc_pos_cash_val"
    ]

    for col in infra_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")  # NaN stays NaN

    for col in transaction_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)  # missing = no activity = 0

        
    # Step 9: Calculate derived spend columns
    # Total CC Spend = PoS + Online + Others (excludes ATM cash withdrawal)
    # This is the unified metric consistent across all months
    df["cc_total_spend_vol"] = df["cc_pos_vol"]  + df["cc_online_vol"]  + df["cc_others_vol"]
    df["cc_total_spend_val"] = df["cc_pos_val"]  + df["cc_online_val"]  + df["cc_others_val"]

    all_months.append(df)
    print(f"✓  {filepath.name}  →  {len(df)} banks")

✓  rbi_2022_03.XLSX  →  58 banks
✓  rbi_2022_04.XLSX  →  58 banks
✓  rbi_2022_05.XLSX  →  58 banks
✓  rbi_2022_06.XLSX  →  58 banks
✓  rbi_2022_07.XLSX  →  58 banks
✓  rbi_2022_08.XLSX  →  58 banks
✓  rbi_2022_09.XLSX  →  58 banks
✓  rbi_2022_10.XLSX  →  60 banks
✓  rbi_2022_11.XLSX  →  61 banks
✓  rbi_2022_12.XLSX  →  61 banks
✓  rbi_2023_01.XLSX  →  61 banks
✓  rbi_2023_02.XLSX  →  61 banks
✓  rbi_2023_03.XLSX  →  61 banks
✓  rbi_2023_04.XLSX  →  61 banks
✓  rbi_2023_05.XLSX  →  61 banks


c:\Users\tanuj\Desktop\Projects\rbi_credit_card_project\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\tanuj\Desktop\Projects\rbi_credit_card_project\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


✓  rbi_2023_06.XLSX  →  61 banks
✓  rbi_2023_07.XLSX  →  61 banks
✓  rbi_2023_08.XLSX  →  61 banks
✓  rbi_2023_09.XLSX  →  61 banks
✓  rbi_2023_10.XLSX  →  61 banks
✓  rbi_2023_11.XLSX  →  61 banks
✓  rbi_2023_12.XLSX  →  65 banks
✓  rbi_2024_01.XLSX  →  65 banks
✓  rbi_2024_02.XLSX  →  65 banks
✓  rbi_2024_03.XLSX  →  65 banks
✓  rbi_2024_04.XLSX  →  64 banks
✓  rbi_2024_05.XLSX  →  64 banks
✓  rbi_2024_06.XLSX  →  64 banks
✓  rbi_2024_07.XLSX  →  64 banks
✓  rbi_2024_08.XLSX  →  64 banks
✓  rbi_2024_09.XLSX  →  64 banks
✓  rbi_2024_10.XLSX  →  64 banks
✓  rbi_2024_11.XLSX  →  64 banks
✓  rbi_2024_12.XLSX  →  64 banks
✓  rbi_2025_01.XLSX  →  64 banks
✓  rbi_2025_02.XLSX  →  64 banks
✓  rbi_2025_03.XLSX  →  64 banks
✓  rbi_2025_04.XLSX  →  64 banks
✓  rbi_2025_05.XLSX  →  64 banks
✓  rbi_2025_06.XLSX  →  64 banks
✓  rbi_2025_07.XLSX  →  64 banks
✓  rbi_2025_08.XLSX  →  64 banks
✓  rbi_2025_09.XLSX  →  64 banks
✓  rbi_2025_10.XLSX  →  64 banks
✓  rbi_2025_11.XLSX  →  64 banks
✓  rbi_202

In [44]:
master_df = pd.concat(all_months, ignore_index=True)

# Reorder columns — date and identifiers first, then metrics
final_cols = [
    "year", "month", "bank_name_std", "bank_category",
    "cc_outstanding", "dc_outstanding",
    "cc_pos_vol", "cc_pos_val",
    "cc_online_vol", "cc_online_val",
    "cc_others_vol", "cc_others_val",
    "cc_atm_cash_vol", "cc_atm_cash_val",
    "cc_total_spend_vol", "cc_total_spend_val",
    "dc_pos_vol", "dc_pos_val",
    "dc_online_vol", "dc_online_val",
    "dc_atm_cash_vol", "dc_atm_cash_val",
    "dc_pos_cash_vol", "dc_pos_cash_val",
    "atm_onsite", "atm_offsite", "pos_terminals",
    "micro_atms", "bharat_qr", "upi_qr"
]

master_df[final_cols].to_csv(PROCESSED / "master_data.csv", index=False)

# Correct date range: find earliest and latest year-month combinations
sorted_ym = master_df[['year','month']].drop_duplicates().sort_values(['year','month'])
earliest = sorted_ym.iloc[0]
latest   = sorted_ym.iloc[-1]

print(f"master_data.csv saved")
print(f"Total rows    : {len(master_df)}")
print(f"Total columns : {len(final_cols)}")
print(f"Months covered: {int(earliest['year'])}-{int(earliest['month']):02d}  to  {int(latest['year'])}-{int(latest['month']):02d}")
print(f"Unique banks  : {master_df['bank_name_std'].nunique()}")
print(f"Unique months : {master_df[['year','month']].drop_duplicates().shape[0]}")


master_data.csv saved
Total rows    : 3118
Total columns : 30
Months covered: 2022-03  to  2026-04
Unique banks  : 66
Unique months : 50


In [45]:
# Check every month is present — no gaps
all_months_check = master_df.groupby(["year","month"])["bank_name_std"].count().reset_index()
all_months_check.columns = ["year", "month", "bank_count"]
print(all_months_check.to_string())
# Visually scan: every month should have 55-65 banks
# Any month with significantly fewer = something was dropped incorrectly

# Export unmatched names so you can add them to bank_name_master.xlsx
if unmatched:
    unmatched_df = pd.concat(unmatched).drop_duplicates(subset="bank_name")
    unmatched_df.to_csv(PROCESSED / "unmatched_banks_log.csv", index=False)
    print(f"\n⚠  {len(unmatched_df)} unmatched bank names — add to master and rerun")
else:
    print("\n✓  All bank names matched")

    year  month  bank_count
0   2022      3          58
1   2022      4          58
2   2022      5          58
3   2022      6          58
4   2022      7          58
5   2022      8          58
6   2022      9          58
7   2022     10          60
8   2022     11          61
9   2022     12          61
10  2023      1          61
11  2023      2          61
12  2023      3          61
13  2023      4          61
14  2023      5          61
15  2023      6          61
16  2023      7          61
17  2023      8          61
18  2023      9          61
19  2023     10          61
20  2023     11          61
21  2023     12          65
22  2024      1          65
23  2024      2          65
24  2024      3          65
25  2024      4          64
26  2024      5          64
27  2024      6          64
28  2024      7          64
29  2024      8          64
30  2024      9          64
31  2024     10          64
32  2024     11          64
33  2024     12          64
34  2025      1     

In [48]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from pathlib import Path

# Load the clean master data
master_df = pd.read_csv(Path("../processed/master_data.csv"))

# Connect to PostgreSQL — update password if yours is different
conn = psycopg2.connect(
    host     = "localhost",
    port     = 5432,
    database = "rbi_credit_card",
    user     = "postgres",
    password = "1234"        # replace with your actual password
)
cur = conn.cursor()

# Create table — drop first if rerunning
cur.execute("DROP TABLE IF EXISTS card_data;")

cur.execute("""
    CREATE TABLE card_data (
        year                 INTEGER,
        month                INTEGER,
        bank_name_std        TEXT,
        bank_category        TEXT,
        cc_outstanding       NUMERIC,
        dc_outstanding       NUMERIC,
        cc_pos_vol           NUMERIC,
        cc_pos_val           NUMERIC,
        cc_online_vol        NUMERIC,
        cc_online_val        NUMERIC,
        cc_others_vol        NUMERIC,
        cc_others_val        NUMERIC,
        cc_atm_cash_vol      NUMERIC,
        cc_atm_cash_val      NUMERIC,
        cc_total_spend_vol   NUMERIC,
        cc_total_spend_val   NUMERIC,
        dc_pos_vol           NUMERIC,
        dc_pos_val           NUMERIC,
        dc_online_vol        NUMERIC,
        dc_online_val        NUMERIC,
        dc_atm_cash_vol      NUMERIC,
        dc_atm_cash_val      NUMERIC,
        dc_pos_cash_vol      NUMERIC,
        dc_pos_cash_val      NUMERIC,
        atm_onsite           NUMERIC,
        atm_offsite          NUMERIC,
        pos_terminals        NUMERIC,
        micro_atms           NUMERIC,
        bharat_qr            NUMERIC,
        upi_qr               NUMERIC
    );
""")

# Load rows one by one
# NaN in dataframe must become NULL in PostgreSQL — replace NaN with None first
master_df = master_df.where(pd.notna(master_df), None)

rows = [tuple(row) for row in master_df.itertuples(index=False)]

insert_query = """
    INSERT INTO card_data VALUES (
        %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
        %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
        %s,%s,%s,%s,%s,%s,%s,%s,%s,%s
    )
"""

cur.executemany(insert_query, rows)
conn.commit()

# Verify
cur.execute("SELECT COUNT(*) FROM card_data;")
count = cur.fetchone()[0]
print(f"Rows loaded into PostgreSQL: {count}")

cur.execute("SELECT MIN(year), MIN(month), MAX(year), MAX(month) FROM card_data;")
print(f"Date range in DB: {cur.fetchone()}")

cur.execute("SELECT COUNT(DISTINCT bank_name_std) FROM card_data;")
print(f"Unique banks: {cur.fetchone()[0]}")

cur.close()
conn.close()
print("Connection closed. Table ready.")

Rows loaded into PostgreSQL: 3118
Date range in DB: (2022, 1, 2026, 12)
Unique banks: 66
Connection closed. Table ready.
